In [2]:
!pip install pandas sqlalchemy psycopg2-binary

In [ ]:
import pandas as pd
from sqlalchemy import create_engine

DB_USER = "postgres"
DB_PASSWORD = "xxxxxxxxxx"
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "calgary_healthcare"

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

print("Database connection created.")

Database connection created.


In [5]:
from pathlib import Path

BASE_DIR = Path.cwd()
RAW_DIR = BASE_DIR / "data" / "raw"

In [18]:
import pandas as pd
import numpy as np
from faker import Faker
from pathlib import Path

fake = Faker()
np.random.seed(42)

# Folder to save files
output = Path("data/raw")
output.mkdir(parents=True, exist_ok=True)

# -----------------------------
# 1. PATIENTS
# -----------------------------

patients = pd.DataFrame({
    "patient_id": [f"P{i:05d}" for i in range(1, 1001)],
    "gender": np.random.choice(["Male", "Female"], 1000),
    "age": np.random.randint(1, 90, 1000),
    "postal_prefix": np.random.choice(
        ["T2A", "T2B", "T2G", "T2J", "T2N", "T2R", "T2T", "T3A"],
        1000
    )
})

# -----------------------------
# 2. HOSPITALS
# -----------------------------

hospitals = pd.DataFrame({
    "hospital_id": ["H001", "H002", "H003", "H004", "H005"],
    "hospital_name": [
        "Calgary Central Hospital",
        "Calgary North Medical Centre",
        "Calgary South Hospital",
        "Calgary Community Care Centre",
        "Calgary Specialty Centre"
    ],
    "facility_type": [
        "Acute Care",
        "Acute Care",
        "Acute Care",
        "Community",
        "Specialty"
    ]
})

# -----------------------------
# 3. PROVIDERS
# -----------------------------

providers = pd.DataFrame({
    "provider_id": [f"PR{i:03d}" for i in range(1, 101)],
    "specialty": np.random.choice(
        ["Cardiology", "Neurology", "Orthopedics",
         "Oncology", "Emergency", "Family Medicine"],
        100
    ),
    "hospital_id": np.random.choice(
        hospitals["hospital_id"], 100
    )
})

# -----------------------------
# 4. ENCOUNTERS
# -----------------------------

encounters = pd.DataFrame({
    "encounter_id": [f"E{i:05d}" for i in range(1, 3001)],
    "patient_id": np.random.choice(
        patients["patient_id"], 3000
    ),
    "hospital_id": np.random.choice(
        hospitals["hospital_id"], 3000
    ),
    "provider_id": np.random.choice(
        providers["provider_id"], 3000
    ),
    "encounter_type": np.random.choice(
        ["Emergency", "Inpatient", "Outpatient",
         "Day Surgery", "Diagnostic Imaging"],
        3000
    ),
    "department": np.random.choice(
        ["Emergency", "Cardiology", "Surgery",
         "Radiology", "General Medicine"],
        3000
    ),
    "diagnosis_code": np.random.choice(
        ["I10", "E11.9", "J18.9", "M54.5", "J45.9"],
        3000
    )
})

# -----------------------------
# 5. CLAIMS
# -----------------------------

claims = pd.DataFrame({
    "claim_id": [f"C{i:05d}" for i in range(1, 5001)],
    "encounter_id": np.random.choice(
        encounters["encounter_id"], 5000
    ),
    "claim_date": pd.date_range(
        "2025-01-01", periods=5000, freq="3h"
    ).date,
    "claim_type": np.random.choice(
        ["Emergency", "Inpatient", "Outpatient",
         "Surgery", "Diagnostic"],
        5000
    ),
    "service_code": np.random.choice(
        ["SRV001", "SRV002", "SRV003",
         "SRV004", "SRV005"],
        5000
    ),
    "billed_amount": np.round(
        np.random.uniform(100, 15000, 5000), 2
    ),
    "claim_status": np.random.choice(
        ["Paid", "Pending", "Denied",
         "Rejected", "Partially Paid"],
        5000,
        p=[0.55, 0.10, 0.10, 0.10, 0.15]
    )
})

# Approved amount
claims["approved_amount"] = np.round(
    claims["billed_amount"] * np.random.uniform(0.7, 1.0, 5000),
    2
)

# Paid amount
claims["paid_amount"] = np.where(
    claims["claim_status"] == "Paid",
    claims["approved_amount"],
    np.where(
        claims["claim_status"] == "Partially Paid",
        claims["approved_amount"] * np.random.uniform(0.3, 0.8, 5000),
        0
    )
)

claims["paid_amount"] = claims["paid_amount"].round(2)

# Processing time
claims["processing_days"] = np.random.randint(1, 46, 5000)

# Denial reason
claims["denial_code"] = np.where(
    claims["claim_status"].isin(["Denied", "Rejected"]),
    np.random.choice(
        ["D001", "D002", "D003", "D004", "D005"],
        5000
    ),
    None
)

# -----------------------------
# 6. DENIAL REASONS
# -----------------------------

denials = pd.DataFrame({
    "denial_code": [
        "D001", "D002", "D003", "D004", "D005"
    ],
    "denial_reason": [
        "Missing documentation",
        "Invalid service code",
        "Duplicate claim",
        "Eligibility issue",
        "Missing authorization"
    ],
    "category": [
        "Documentation",
        "Coding",
        "Billing",
        "Eligibility",
        "Authorization"
    ]
})

# -----------------------------
# 7. CLAIM TRANSACTIONS
# -----------------------------

transactions = pd.DataFrame({
    "transaction_id": [f"T{i:06d}" for i in range(1, 10001)],
    "claim_id": np.random.choice(
        claims["claim_id"], 10000
    ),
    "transaction_type": np.random.choice(
        ["Submission", "Adjustment", "Payment", "Resubmission"],
        10000
    ),
    "transaction_date": pd.date_range(
        "2025-01-01", periods=10000, freq="90min"
    ).date,
    "paid_amount": np.round(
        np.random.uniform(0, 5000, 10000), 2
    )
})

# -----------------------------
# 8. SAVE FILES
# -----------------------------

datasets = {
    "patients": patients,
    "hospitals": hospitals,
    "providers": providers,
    "encounters": encounters,
    "claims": claims,
    "claim_transactions": transactions,
    "denial_reasons": denials
}

for name, df in datasets.items():
    df.to_csv(output / f"{name}.csv", index=False)

print("Synthetic healthcare data created successfully!")

for name, df in datasets.items():
    print(f"{name}.csv: {len(df):,} rows")

Synthetic healthcare data created successfully!
patients.csv: 1,000 rows
hospitals.csv: 5 rows
providers.csv: 100 rows
encounters.csv: 3,000 rows
claims.csv: 5,000 rows
claim_transactions.csv: 10,000 rows
denial_reasons.csv: 5 rows


In [20]:
# To load the datasets from the raw CSV files into pandas DataFrames

patients = pd.read_csv(RAW_DIR / "patients.csv")
hospitals = pd.read_csv(RAW_DIR / "hospitals.csv")
providers = pd.read_csv(RAW_DIR / "providers.csv")
encounters = pd.read_csv(RAW_DIR / "encounters.csv")
claims = pd.read_csv(RAW_DIR / "claims.csv")
claim_transactions = pd.read_csv(
    RAW_DIR / "claim_transactions.csv"
)
denial_reasons = pd.read_csv(
    RAW_DIR / "denial_reasons.csv"
)

In [22]:

# To check the number of rows and columns in each dataset
datasets = {
    "patients": patients,
    "hospitals": hospitals,
    "providers": providers,
    "encounters": encounters,
    "claims": claims,
    "claim_transactions": claim_transactions,
    "denial_reasons": denial_reasons
}

for name, df in datasets.items():
    print(f"\n{name}")
    print(f"Rows: {len(df)}")
    print(f"Columns: {len(df.columns)}")


patients
Rows: 1000
Columns: 4

hospitals
Rows: 5
Columns: 3

providers
Rows: 100
Columns: 3

encounters
Rows: 3000
Columns: 7

claims
Rows: 5000
Columns: 11

claim_transactions
Rows: 10000
Columns: 5

denial_reasons
Rows: 5
Columns: 3


In [23]:
# To check for duplicate patient records in the patients dataset

duplicates = patients[
    patients["patient_id"].duplicated()
]

print(
    f"Duplicate patients: {len(duplicates)}"
)

Duplicate patients: 0


In [24]:
# To check for duplicate claim records in the claims dataset

duplicates = claims[
    claims["claim_id"].duplicated()
]

print(
    f"Duplicate claims: {len(duplicates)}"
)

Duplicate claims: 0


In [28]:
df = pd.read_csv(RAW_DIR / "patients.csv")
df

,patient_id,gender,age,postal_prefix
0,P00001,Male,63,T2R
1,P00002,Female,17,T2G
2,P00003,Male,73,T2G
3,P00004,Male,33,T2G
4,P00005,Male,84,T2A
...,...,...,...,...
995,P00996,Male,32,T2B
996,P00997,Male,2,T2N
997,P00998,Female,47,T2T
998,P00999,Female,78,T3A


In [30]:
df.isnull().sum()

np.int64(0)

In [31]:

# To convert the specified columns in the claims dataset to numeric types, coercing errors to NaN
numeric_columns = [
    "billed_amount",
    "approved_amount",
    "paid_amount",
    "processing_days"
]

for column in numeric_columns:
    claims[column] = pd.to_numeric(
        claims[column],
        errors="coerce"
    )

In [34]:
# To convert the claim_date and payment_date columns in the claims dataset to datetime types, coercing errors to NaT

claims["claim_date"] = pd.to_datetime(
    claims["claim_date"],
    errors="coerce"
)

pass


In [ ]:
# Convert the admission_date and discharge_date columns in the encounters dataset to datetime types, coercing errors to NaT

encounters["admission_date"] = pd.to_datetime(
    encounters.get("admission_date", pd.NaT),
    errors="coerce"
)

encounters["discharge_date"] = pd.to_datetime(
    encounters.get("discharge_date", pd.NaT),
    errors="coerce"
)

In [39]:
# To validate claims amounts for negative values

invalid_amounts = claims[
    (claims["billed_amount"] < 0) |
    (claims["approved_amount"] < 0) |
    (claims["paid_amount"] < 0)
]

print(
    f"Invalid monetary records: {len(invalid_amounts)}"
)

Invalid monetary records: 0


In [40]:
# To check if the paid amount exceeds the billed amount

paid_exceeds_billed = claims[
    claims["paid_amount"] > claims["billed_amount"]
]

print(
    f"Paid amount > billed amount: "
    f"{len(paid_exceeds_billed)}"
)

Paid amount > billed amount: 0


In [ ]:
# To create and save the processed data directory

PROCESSED_DIR = BASE_DIR / "data" / "processed"

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [42]:
patients.to_csv(
    PROCESSED_DIR / "patients.csv",
    index=False
)

hospitals.to_csv(
    PROCESSED_DIR / "hospitals.csv",
    index=False
)

providers.to_csv(
    PROCESSED_DIR / "providers.csv",
    index=False
)

encounters.to_csv(
    PROCESSED_DIR / "encounters.csv",
    index=False
)

claims.to_csv(
    PROCESSED_DIR / "claims.csv",
    index=False
)

claim_transactions.to_csv(
    PROCESSED_DIR / "claim_transactions.csv",
    index=False
)

denial_reasons.to_csv(
    PROCESSED_DIR / "denial_reasons.csv",
    index=False
)

In [43]:
# Loading patients data into the staging schema of the database

patients.to_sql(
    "patients",
    engine,
    schema="staging",
    if_exists="append",
    index=False
)

1000

In [ ]:
# Loading hospitals, providers, encounters, claims, claim transactions, and denial reasons data into the staging schema of the database

hospitals.to_sql(
    "hospitals",
    engine,
    schema="staging",
    if_exists="append",
    index=False
)

providers.to_sql(
    "providers",
    engine,
    schema="staging",
    if_exists="append",
    index=False
)

encounters.to_sql(
    "encounters",
    engine,
    schema="staging",
    if_exists="append",
    index=False
)

claims.to_sql(
    "claims",
    engine,
    schema="staging",
    if_exists="append",
    index=False
)

claim_transactions.to_sql(
    "claim_transactions",
    engine,
    schema="staging",
    if_exists="append",
    index=False
)

denial_reasons.to_sql(
    "denial_reasons",
    engine,
    schema="staging",
    if_exists="append",
    index=False
)

5